In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)

In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
from openbabel import pybel
import shutil
import time

In [ ]:

MAX_TASKS = 2

In [ ]:

NPROC = 10
MEM = 20  

In [ ]:

directory = "energy"
opt_success_directory = "opt+freq/success"
failure_path = os.path.join(directory, "failure")
success_path = os.path.join(directory, "success")

In [ ]:

key_words_for_Convergence_failure = "# M062X/6-311+G(2d,p) em=gd3 SCF=(novaracc,noincfock,conver=6,vshift=400)"

In [ ]:

def convert_out_to_gjf(out_file, gjf_file):
    
    subprocess.run(["obabel", "-i", "out", out_file, "-o", "gjf", "-O", gjf_file])

In [ ]:


def extract_charge_and_coordinates(gjf_file):
    try:
        with open(gjf_file, 'r') as file:
            lines = file.readlines()
    except FileNotFoundError:
        return None, f"File {gjf_file} not found."

    if not lines:
        return None, f"File {gjf_file} is empty."

    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*')

    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index - 1
            break

    if start_index is None:
        return None, f"No atomic coordinates found in {gjf_file}"

    
    final_index = start_index + 1
    for index, line in enumerate(lines[start_index + 1:], start=start_index + 1):
        if not atom_line_pattern.match(line):
            final_index = index
            break

    
    charge_and_multiplicity_line = lines[start_index]
    coordinates_lines = lines[start_index + 1 : final_index]

    return (charge_and_multiplicity_line, coordinates_lines), None

In [ ]:


def extract_smd_parameters(gjf_file):
    smd_parameters = []
    with open(gjf_file, 'r') as file:
        lines = file.readlines()
    
    
    smd_start_found = False
    for line in lines:
        if line.strip().startswith('eps='):
            smd_start_found = True
        if smd_start_found:
            smd_parameters.append(line)
            if line.strip().startswith('ElectronegativeHalogenicity='):
                break
    
    return smd_parameters

In [ ]:

def modify_and_append_to_gjf(gjf_file, new_keywords, chk_directory, charge_and_coords, smd_parameters):
    print(f"Using keywords for {gjf_file}: {new_keywords}")  
    current_dir = os.getcwd()
    chk_file = os.path.join(current_dir, gjf_file.replace('.gjf', '.chk'))
    
    
    base_name = os.path.basename(gjf_file).replace(".gjf", "")
    oldchk_file = base_name + ".chk"
    oldchk_file = os.path.join(current_dir, opt_success_directory, oldchk_file)
    
    with open(gjf_file, 'w') as file:
        file.write(f"%nprocshared={NPROC}\n")
        file.write(f"%mem={MEM}GB\n")
        file.write(f"%oldchk={oldchk_file}\n")
        file.write(f'%chk={chk_file}\n')
        file.write(f'{new_keywords}\n\n')
        file.write('dimer energy \n\n')
        file.write(charge_and_coords[0])  
        file.writelines(charge_and_coords[1])  
        file.write('\n\n')

In [ ]:

def run_gaussian_and_categorize(gjf_file, success_dir, failure_dir):
    base_name = os.path.basename(gjf_file).replace(".gjf", "")
    out_file = base_name + ".out"
    chk_file = base_name + ".chk"
    
    
    try:
        subprocess.run(["g16", gjf_file, out_file])

        
        with open(out_file, 'r') as file:
            contents = file.read()
            if "Normal termination" in contents:
                
                os.rename(gjf_file, os.path.join(success_dir, os.path.basename(gjf_file)))
                os.rename(out_file, os.path.join(success_dir, out_file))
                os.rename(chk_file, os.path.join(success_dir, chk_file))
                return out_file, True
            else:
                
                os.rename(gjf_file, os.path.join(failure_dir, os.path.basename(gjf_file)))
                os.rename(out_file, os.path.join(failure_dir, out_file))
                os.rename(chk_file, os.path.join(failure_dir, chk_file))
    except subprocess.CalledProcessError:
        
        os.rename(gjf_file, os.path.join(failure_dir, gjf_file))
        if os.path.exists(out_file):
            os.rename(out_file, os.path.join(failure_dir, out_file))
        if os.path.exists(chk_file):
            os.rename(chk_file, os.path.join(failure_dir, chk_file))

    return None, False    

In [ ]:
def check_and_move_files(directory, success_path, failure_path):
    
    
    moved_to_success = []
    moved_to_failure = []

    
    for file in os.listdir(directory):
        if file.endswith(".out"):  
            file_path = os.path.join(directory, file)
            with open(file_path, 'r') as f:
                content = f.read()
                
                if "Normal termination" in content:
                    
                    target_dir = success_path
                    moved_to_success.append(file)  
                else:
                    
                    target_dir = failure_path
                    moved_to_failure.append(file)  
                
                
                base_filename = os.path.splitext(file)[0]
                for ext in ['.gjf', '.chk', '.out']:
                    src_file = os.path.join(directory, base_filename + ext)
                    if os.path.exists(src_file):  
                        dst_file = os.path.join(target_dir, base_filename + ext)
                        os.rename(src_file, dst_file)
    
    
    print("Moved to success folder:")
    for filename in set(moved_to_success):  
        print(os.path.splitext(filename)[0])

    print("Moved to failure folder:")
    for filename in set(moved_to_failure):  
        print(os.path.splitext(filename)[0])

In [ ]:

def clean_up_failures(success_path, failure_path):
    
    success_files = set()
    for filename in os.listdir(success_path):
        name, ext = os.path.splitext(filename)
        success_files.add(name)

    
    for filename in os.listdir(failure_path):
        name, ext = os.path.splitext(filename)

        
        if name in success_files:
            for ext in ['.gjf', '.chk', '.out']:
                file_to_delete = os.path.join(failure_path, name + ext)
                if os.path.exists(file_to_delete):
                    os.remove(file_to_delete)
                    print(name)

In [ ]:

def main():
    
    
    start_time = time.time() 
    
    
    for out_file in os.listdir(failure_path):
        if out_file.endswith(".out"):
            
            source_file = os.path.join(failure_path, out_file)
            target_file = os.path.join(directory, out_file.replace(".out", ".gjf"))
            
            
            base_filename = os.path.splitext(out_file)[0]

            
            convert_out_to_gjf(source_file, target_file)

            
            charge_and_coords, error_message  = extract_charge_and_coordinates(target_file)
            if error_message:
                print(error_message)
                continue  

            
            smd_parameters = extract_smd_parameters(source_file)
            
            
            modify_and_append_to_gjf(target_file, key_words_for_Convergence_failure, directory, charge_and_coords, smd_parameters)
            
            
            print(f"Convergence failure found in {source_file}, using key_words_for_Convergence_failure.")
            
            
                          
    
    with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
        futures = [executor.submit(run_gaussian_and_categorize, os.path.join(directory, f), success_path, failure_path) for f in os.listdir(directory) if f.endswith(".gjf")]
        for future in as_completed(futures):
            pass  
    
    check_and_move_files(directory, success_path, failure_path)
    clean_up_failures(success_path, failure_path)
    
    
    end_time = time.time()
    
    
    elapsed_time = end_time - start_time
    print(f"The code ran for {elapsed_time} seconds.")


In [ ]:
if __name__ == "__main__":
    main()